# 8B LoRA 파일럿 (Colab·Kaggle 노트북용)

`run.sh`와 같은 순서를 셀로 나눈 것. 학습 자료 `train-8k.jsonl`(노트북에서 `subsample.py`로 생성)을 먼저 올린다.
- Colab: 왼쪽 파일 창에 `train-8k.jsonl`을 끌어다 놓거나 Google Drive에 두고 아래 경로를 맞춘다. 런타임 유형은 GPU(A100 또는 L4).
- Kaggle: 학습 자료를 Dataset으로 올리고 Add Input 뒤 `/kaggle/input/<이름>/train-8k.jsonl` 경로를 쓴다. 가속기 GPU T4 x2(16 GB, 4비트 필요).

세션이 끊기면 처음부터 다시 돌아가므로 결과는 셀마다 Drive(또는 /kaggle/working)로 복사한다.

In [ ]:
import os, torch
print(torch.__version__, torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30), 'GB')
Q4 = torch.cuda.get_device_properties(0).total_memory < 20 * 2**30     # 16 GB급이면 4비트
print('4비트:', Q4)

In [ ]:
# Colab이면 Drive를 붙여 결과 보관 (Kaggle은 건너뜀)
IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    SAVE = '/content/drive/MyDrive/lora-pilot'
else:
    SAVE = '/kaggle/working/lora-pilot'
os.makedirs(SAVE, exist_ok=True); print(SAVE)

In [ ]:
!git clone -q https://github.com/kjhoeakjy/multimodal-doc-structuring.git
%cd multimodal-doc-structuring/examples/llm-qa
!pip install -q -r finetune/requirements.txt

In [ ]:
# 학습 자료 경로 (환경에 맞게 하나만 남김)
DATA = '/content/train-8k.jsonl'                       # Colab 파일 창에 올린 경우
# DATA = '/content/drive/MyDrive/train-8k.jsonl'      # Drive에 둔 경우
# DATA = '/kaggle/input/lora-train/train-8k.jsonl'    # Kaggle Dataset
assert os.path.exists(DATA), DATA
MODEL = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
os.environ['HF_TOKEN'] = input('HF 토큰 입력(화면에 남지 않음): ') if not os.environ.get('HF_TOKEN') else os.environ['HF_TOKEN']

In [ ]:
# 학습 (세션 한도 안에 끝나도록 에폭·예시 수를 조절. 끊기면 adapter/checkpoint-*에서 이어서 돌릴 수 있음)
!python finetune/train_lora.py --data {DATA} --model {MODEL} --out finetune/adapter --epochs 2 --max-len 6144 {'--4bit' if Q4 else ''}
!cp -r finetune/adapter {SAVE}/

In [ ]:
# 평가 4회: 기본 가중치·LoRA × 43문항(noise)·36문항(conn), 대표 설정 12개, full·paths
SET = r'^(none|drop|ocr|attr|mix)-(0|0.2|0.5)-|^add-0.5|^real'
q4 = '--4bit' if Q4 else ''
!python finetune/eval_lora.py --model {MODEL} --label open-8b-hf-2026-11-noise --conds full,paths --settings "{SET}" {q4}
!cp results/open-8b-hf-2026-11-noise.* {SAVE}/
!python finetune/eval_lora.py --model {MODEL} --label open-8b-hf-2026-11-conn --qfile questions-conn.json --conds full,paths --settings "{SET}" {q4}
!cp results/open-8b-hf-2026-11-conn.* {SAVE}/
!python finetune/eval_lora.py --model {MODEL} --adapter finetune/adapter --label open-8b-lora-2026-11-noise --conds full,paths --settings "{SET}" {q4}
!cp results/open-8b-lora-2026-11-noise.* {SAVE}/
!python finetune/eval_lora.py --model {MODEL} --adapter finetune/adapter --label open-8b-lora-2026-11-conn --qfile questions-conn.json --conds full,paths --settings "{SET}" {q4}
!cp results/open-8b-lora-2026-11-conn.* {SAVE}/
print('완료:', os.listdir(SAVE))